# Investment AI
## Notebook 07 — Build the Research Agent

The retrieval layer is now working.

This notebook gives a Claude-powered Research Agent access to that retrieval layer so it can:

1. understand a research question,
2. decide when research evidence is needed,
3. call a governed Databricks AI Search tool,
4. reason over the retrieved evidence,
5. return an evidence-based answer with source citations.

```text
User Question
     ↓
Research Agent loop
     ↓
Claude Sonnet 5.5
     ↓
Unity Gateway Model Provider Service
     ↓
Anthropic Claude
     ↓
Claude requests research tool when needed
     ↓
AI Bridge Vector Search Retriever
     ↓
Databricks AI Search
     ↓
Governed Research Evidence
     ↓
Evidence returned to Claude
     ↓
Final Answer + Citations
```

The Research Agent is intentionally scoped to **unstructured investment research**.

It does not answer portfolio-position, ownership, or exposure questions. Those will belong to the Portfolio Agent later.


## Why AI Bridge here?

For new Python-based AI Search integrations, we use the **AI Bridge retriever tool** to query our existing Databricks AI Search index directly from Python.

We will use:

- `VectorSearchRetrieverTool` from `databricks-openai`
- Anthropic's native Python SDK for Claude
- Anthropic's native **tool-use** pattern for the agent loop
- Claude Sonnet 5.5 through **Unity Gateway**
- `investment_ai.gateway.anthropic_claude` as our governed Model Provider Service
- Hybrid retrieval as the default search strategy

The Anthropic API key remains behind Unity Gateway and is not stored in this notebook.

The AI Search index remains governed through Unity Catalog.


In [0]:
%pip install -q --upgrade databricks-openai anthropic "mlflow[databricks]>=3.1.0"
dbutils.library.restartPython()


In [0]:
import mlflow

# Use Databricks as the MLflow tracking backend
mlflow.set_tracking_uri("databricks")

# Use a stable experiment instead of the notebook's temporary experiment
mlflow.set_experiment(
    "/Shared/investment_ai_research_agent"
)

# Show traces directly inside notebook output
mlflow.tracing.enable_notebook_display()

# Automatically trace Anthropic / Claude API calls
mlflow.anthropic.autolog()

print("✅ MLflow tracing configured.")
print("Experiment: /Shared/investment_ai_research_agent")

In [0]:
import json

from anthropic import AsyncAnthropic
from databricks.sdk import WorkspaceClient

from databricks_openai import VectorSearchRetrieverTool

AI_SEARCH_INDEX = (
    "investment_ai.research.investment_ai_research_index"
)

MODEL_PROVIDER_SERVICE = (
    "investment_ai.gateway.anthropic_claude"
)

# Production-quality reasoning for the Research Agent.
RESEARCH_MODEL = "claude-sonnet-5-5"

# Cheaper option while developing/testing:
# RESEARCH_MODEL = "claude-haiku-5-5"

RETURN_COLUMNS = [
    "chunk_to_retrieve",
    "security_id",
    "company_name",
    "document_type",
    "source_type",
    "source_authority_rank",
    "publish_date",
    "citation_label",
]

print("Research model        :", RESEARCH_MODEL)
print("Model provider service:", MODEL_PROVIDER_SERVICE)
print("AI Search index       :", AI_SEARCH_INDEX)


## 1. Create the research retrieval capability

This is the bridge between the agent and our AI Search index.

The agent does not read the Delta table directly.

Instead, it calls a retrieval tool that searches the governed AI Search index using **Hybrid Search** and returns the most relevant evidence.

Optional metadata filters let the agent narrow retrieval when it already knows the security, quarter, year, or preferred source type.


In [0]:
def _run_research_search(
    query: str,
    security_id: str | None = None,
    fiscal_quarter: str | None = None,
    fiscal_year: int | None = None,
    source_type: str | None = None,
):
    filters = {}

    if security_id:
        filters["security_id"] = security_id

    if fiscal_quarter:
        filters["fiscal_quarter"] = fiscal_quarter

    if fiscal_year:
        filters["fiscal_year"] = fiscal_year

    if source_type:
        filters["source_type"] = source_type

    retriever = VectorSearchRetrieverTool(
        index_name=AI_SEARCH_INDEX,
        tool_name="investment_research_search",
        tool_description=(
            "Search the governed Investment AI research corpus. "
            "Use it for earnings, guidance, outlook, management commentary, "
            "business drivers, risks, and other company-research questions."
        ),
        num_results=5,
        columns=RETURN_COLUMNS,
        filters=filters or None,
        query_type="HYBRID",
    )

    return retriever.execute(query=query)


print("✅ Research retrieval capability created.")


## 2. Test retrieval before giving it to the agent

We already tested AI Search directly in Notebook 06.

Here we make one quick call through the same retrieval abstraction the agent will use.

This keeps retrieval and reasoning separate:

```text
Retriever → finds evidence
Agent     → reasons over evidence
```


In [0]:
retrieval_preview = _run_research_search(
    query="Why was the full-year outlook reduced?",
    security_id="IA-SYN-01",
    fiscal_quarter="Q2",
    fiscal_year=2026,
    source_type="primary",
)

print(retrieval_preview)


## 3. Expose retrieval as a Claude tool

Our Python retrieval function works, but Claude still needs a formal description of that capability before it can call it.

So we define an **Anthropic-compatible tool schema** that tells Claude:

- the tool name,
- what the tool is used for,
- which parameters are available,
- which parameters are optional,
- and that `query` is required.

Claude can then decide:

- whether a search is needed,
- what query to send,
- whether useful metadata filters are known,
- and how to use the returned evidence in the final answer.


In [0]:
def search_investment_research(
    query: str,
    security_id: str | None = None,
    fiscal_quarter: str | None = None,
    fiscal_year: int | None = None,
    source_type: str | None = None,
) -> str:
    """Search the governed synthetic investment research corpus.

    Use this tool for company outlook, guidance, earnings commentary,
    business drivers, risks, management statements, and research evidence.
    """

    results = _run_research_search(
        query=query,
        security_id=security_id,
        fiscal_quarter=fiscal_quarter,
        fiscal_year=fiscal_year,
        source_type=source_type,
    )

    return json.dumps(
        results,
        default=str,
    )


SEARCH_TOOL_DEFINITION = {
    "name": "search_investment_research",
    "description": (
        "Search the governed synthetic investment research corpus. "
        "Use this tool for company outlook, guidance, earnings commentary, "
        "business drivers, risks, management statements, and research evidence."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The research question or search query.",
            },
            "security_id": {
                "type": "string",
                "description": "Optional security ID to filter results.",
            },
            "fiscal_quarter": {
                "type": "string",
                "description": "Optional fiscal quarter filter (e.g., Q1, Q2).",
            },
            "fiscal_year": {
                "type": "integer",
                "description": "Optional fiscal year filter.",
            },
            "source_type": {
                "type": "string",
                "description": "Optional source type filter (primary, internal, secondary).",
            },
        },
        "required": ["query"],
    },
}

print("✅ AI Search is now available as an agent tool.")


## 4. Configure Claude through Unity Gateway

The Research Agent uses the Anthropic Claude models that we already registered behind our Unity Gateway Model Provider Service:

```text
investment_ai.gateway.anthropic_claude
```

Because this is an **Anthropic Model Provider Service**, we call it through Anthropic's native **Messages API** rather than using an OpenAI Chat Completions interface.

Our notebook authenticates to Databricks. Unity Gateway then:

1. authenticates the Databricks caller,
2. selects `investment_ai.gateway.anthropic_claude`,
3. uses the Anthropic credential stored behind the provider service,
4. forwards the native Anthropic request to the selected Claude model.




In [0]:
workspace = WorkspaceClient()

# Notebook-native Databricks authentication.
auth_header = workspace.config.authenticate()["Authorization"]
databricks_token = auth_header.replace("Bearer ", "", 1)

UNITY_GATEWAY_ANTHROPIC_URL = (
    f"{workspace.config.host.rstrip('/')}/ai-gateway/anthropic"
)

# Anthropic's native Messages API endpoint on Unity Gateway.
# Unity Gateway does NOT support OpenAI Chat Completions format
# for Anthropic providers, so we use the Anthropic SDK directly.
anthropic_client = AsyncAnthropic(
    auth_token=databricks_token,
    base_url=UNITY_GATEWAY_ANTHROPIC_URL,
    default_headers={
        "Databricks-Model-Provider-Service": MODEL_PROVIDER_SERVICE,
    },
)

print("✅ Claude configured through Unity Gateway.")
print("Gateway :", UNITY_GATEWAY_ANTHROPIC_URL)
print("Provider:", MODEL_PROVIDER_SERVICE)
print("Model   :", RESEARCH_MODEL)


## 5. Define the Research Agent

The instructions are part of the agent's behavior contract.

A few rules are especially important:

1. Use retrieval for factual company-research claims.
2. Prefer primary evidence when sources conflict.
3. Cite the retrieved source labels.
4. Do not invent facts that were not retrieved.
5. Stay within research scope.
6. Do not answer portfolio ownership or exposure questions.

That last boundary becomes important when we introduce the Portfolio Agent and our **Multi-Agent Orchestrator** later.


In [0]:
RESEARCH_AGENT_INSTRUCTIONS = """
You are the Investment AI Research Agent.

Your responsibility is unstructured investment research only.

Use the search_investment_research tool whenever the user asks about:
- company guidance or outlook,
- earnings commentary,
- reasons for business changes,
- company risks or drivers,
- management statements,
- research evidence.

Evidence rules:
- Ground factual research claims in retrieved evidence.
- Prefer primary sources over internal sources, and internal sources over
  secondary sources when evidence conflicts.
- Use citation_label from the retrieved results when citing evidence.
- Do not invent a citation or fact that was not returned by the tool.
- If evidence is incomplete or conflicting, say so clearly.

Scope boundary:
- You do NOT have portfolio holdings, ownership, exposure, or fund-position data.
- If the user asks how much Investment AI owns, explain that portfolio data
  belongs to the Portfolio Agent and should be handled separately.

Keep answers concise, analytical, and evidence-based.
"""

@mlflow.trace(name="research_agent")
async def run_research_agent(user_question: str, max_iterations: int = 10) -> str:
    """Run the Research Agent using Anthropic native tool-use through Unity Gateway."""

    messages = [{"role": "user", "content": user_question}]

    for _ in range(max_iterations):
        response = await anthropic_client.messages.create(
            model=RESEARCH_MODEL,
            max_tokens=4096,
            system=RESEARCH_AGENT_INSTRUCTIONS,
            messages=messages,
            tools=[SEARCH_TOOL_DEFINITION],
        )

        text_parts = []
        tool_results = []
        has_tool_use = False

        for block in response.content:
            if block.type == "text":
                text_parts.append(block.text)
            elif block.type == "tool_use":
                has_tool_use = True
                result = search_investment_research(**block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                })

        if has_tool_use:
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})
        else:
            return "\n".join(text_parts)

    return "Reached maximum tool-call iterations without a final answer."


print("✅ Research Agent created.")


## 6. Ask a focused research question

Now the model is no longer being handed retrieved chunks manually.

It receives the user question and can decide whether it needs to call the AI Search tool.

This is the key transition from a fixed retrieval pipeline:

```text
Question → Retrieve → Prompt → LLM
```

to a tool-using Research Agent:

```text
Question
   ↓
Claude reasons
   ↓
Need evidence?
   ↓
Call research tool
   ↓
Databricks AI Search
   ↓
Evidence returned to Claude
   ↓
Final answer
```


In [0]:
question = (
    "Why did Synthetic Compute Company 01 "
    "lower its outlook in Q2 2026?"
)

result = await run_research_agent(question)

print(result)


## 7. Ask a broader research question

The Research Agent can also search across the corpus and synthesize evidence from multiple companies.

This is the research-only portion of our flagship business question.


In [0]:
question = (
    "Which companies lowered their outlook in Q2 2026, "
    "and why did they lower it? "
    "Use the strongest available research evidence."
)

result = await run_research_agent(question)

print(result)


## 8. Verify the agent boundary

The Research Agent should know what it **cannot** answer.

It can determine:

```text
Which companies lowered outlook?
Why?
What evidence supports that conclusion?
```

It should not determine:

```text
How many shares do we own?
What is our market-value exposure?
Which funds hold the company?
```

Those questions require the structured portfolio layer and will be handled by a separate Portfolio Agent.


In [0]:
boundary_question = (
    "How much do we currently own of the companies "
    "that lowered their outlook?"
)

boundary_result = await run_research_agent(boundary_question)

print(boundary_result)


## Research Agent complete

We now have our first specialized agent:

```text
User
 ↓
Research Agent loop
 ↓
Claude Sonnet 5.5
 ↓
Unity Gateway
 ↓
investment_ai.gateway.anthropic_claude
 ↓
Anthropic Claude
 ↓
Claude requests research tool when needed
 ↓
search_investment_research
 ↓
AI Bridge Vector Search Retriever
 ↓
Databricks AI Search
 ↓
Hybrid semantic + keyword retrieval
 ↓
Governed research evidence
 ↓
Evidence returned to Claude
 ↓
Evidence-based answer + citations
```

The important design principle remains:

> Specialized agents should have the tools and data required for their responsibility — not access to everything.

The Research Agent owns the **unstructured research problem**.

Next, we move to the structured analytics side of the architecture:

**Business Semantics → Metric Views → Genie → Genie Ontology**

Later, our specialized agents will be coordinated through a **custom Multi-Agent Orchestrator** rather than the legacy managed Supervisor Agent pattern.
